# 005 - Level 5

## 5. Window Functions

### 🪟 Funciones de ventana más comunes

- **ROW_NUMBER()**  
  Asigna un **número único** a cada fila.

- **RANK() / DENSE_RANK()**  
  Ordenan las filas según un valor:
  - `RANK()` **salta posiciones** cuando hay empates.
  - `DENSE_RANK()` **no salta posiciones**.

- **LAG()**  
  Permite acceder a los **datos de la fila anterior**.

- **LEAD()**  
  Permite acceder a los **datos de la fila siguiente**.

---

### 🔧 Componentes clave

- **PARTITION BY**  
  Aplica los cálculos **dentro de grupos**.

- **OVER()**  
  Indica a SQL que se está utilizando una **función de ventana**.


In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [2]:
df_matches = pd.read_csv('../Data/WorldCupMatches.csv').rename(columns=lambda col: col.lower())
df_players = pd.read_csv('../Data/WorldCupPlayers.csv').rename(columns=lambda col: col.lower())
df_cups = pd.read_csv('../Data/WorldCups.csv').rename(columns=lambda col: col.lower())

pl_matches = pl.read_csv('../Data/WorldCupMatches.csv').rename(lambda col: col.lower())
pl_players = pl.read_csv('../Data/WorldCupPlayers.csv').rename(lambda col: col.lower())
pl_cups = pl.read_csv('../Data/WorldCups.csv').rename(lambda col: col.lower())

## 🛠️ Ejercicio 5.1: Ranking de Jugadores por Partido

### 📐 La lógica de la numeración (Ranking)

A diferencia de un simple `COUNT`, las **funciones de ventana** permiten asignar un **orden o posición** a cada fila **sin agruparlas**.

En este ejercicio se utilizará **`ROW_NUMBER()`**, que asigna un **número secuencial único** a cada registro dentro de una **ventana** específica.

Para este caso:

- La **ventana** se reinicia cada vez que cambia el `MatchID`.
- El **orden** se determina por el **nombre del jugador**.

---

### 🎯 Objetivo del ejercicio

Generar un **listado de jugadores** donde cada uno tenga un **número de posición** (`1, 2, 3, ...`) dentro de su partido.

Esto nos permitirá responder la siguiente pregunta:

> *¿Quién es el **primer jugador** (en orden alfabético) que aparece en la planilla de cada encuentro?*


```SQL
SELECT
    matchid,
    "Team Initials",
    "Player Name",
    ROW_NUMBER() OVER(
        PARTITION BY matchid
        ORDER BY "Player Name" ASC
        ) AS num_lista
FROM worldcup.players;
```

In [6]:
df_p = df_players.copy()

df_p = df_p.sort_values(by=['matchid', 'player name'])

df_p['num_lista'] = df_p.groupby('matchid').cumcount() + 1

res = df_p[['matchid', 'player name', 'num_lista']]

In [10]:
res = pl_players.sort(['matchid','player name']).with_columns([
    pl.col('player name')
    .cum_count()
    .over('matchid')
    .alias('num_lista')
]).select([
    'matchid', 'player name', 'num_lista'
])


## 🛠️ Ejercicio 5.1.B: Ranking de Goleadores (DENSE_RANK)

### 📐 La lógica del ranking con empates

Para este ejercicio, utilizaremos una **tabla imaginaria** (o el resultado de un `COUNT`) donde se calcula cuántos **goles ha anotado cada jugador**.  
El objetivo es identificar a los **máximos artilleros**.

Imagina el siguiente escenario de goles:

- **Klose**: 16 goles → RANK: 1  
- **Ronaldo**: 15 goles → RANK: 2  
- **Gerd Müller**: 14 goles → RANK: 3  
- **Just Fontaine**: 13 goles → RANK: 4  
- **Lionel Messi**: 13 goles → RANK: 4  
- **Pelé**: 12 goles → ¿Qué número sigue?

#### Diferencia entre RANK() y DENSE_RANK()

- Con **`RANK()`**  
  El siguiente puesto es **6**  
  (se salta el 5 porque hubo dos jugadores en el puesto 4).

- Con **`DENSE_RANK()`**  
  El siguiente puesto es **5**  
  (no deja huecos en la numeración).

---

### 🎯 Objetivo del ejercicio

Crear un **ranking de los años con mayor asistencia de público** (`Attendance`) en la historia de los mundiales, utilizando la tabla `worldcup.cups`.


```SQL
SELECT
    year,
    country,
    attendance,
    DENSE_RANK() OVER(
        ORDER BY REPLACE(attendance,'.','')::INTEGER DESC
        ) AS rank_asistencia
FROM worldcup.cups
```

In [16]:
# 1. Copia
df_c = df_cups.copy()

# 2. Limpieza y conversión (Usando los nombres reales del CSV)
# Quitamos el punto y convertimos. errors='coerce' por si hay datos basura.
df_c['Attendance_int'] = df_c['attendance'].str.replace('.', '', regex=False)
df_c['Attendance_int'] = pd.to_numeric(df_c['Attendance_int'], errors='coerce').fillna(0).astype(int)

# 3. Ranking Denso (DENSE_RANK)
df_c['rank_asistencia'] = df_c['Attendance_int'].rank(method='dense', ascending=False).astype(int)

# 4. Resultado final con nombres correctos
res = df_c[['year', 'country', 'attendance', 'rank_asistencia']].sort_values('rank_asistencia')

In [20]:
res = pl_cups.with_columns([
    pl.col('attendance')
    .str.replace_all(r'\.','')
    .cast(pl.Int64, strict=False)
    .rank('dense', descending=True)
    .alias('rank_asistencia')
]).select([
    'year', 'country', 'attendance', 'rank_asistencia'
]).sort('rank_asistencia')

## 🛠️ Ejercicio 5.2: Comparación Temporal (LAG)

### 📐 La lógica del "Retrovisor"

La función **`LAG()`** actúa como un **retrovisor**:  
te permite estar situado en un año (por ejemplo, **1938**) y ver qué ocurrió en el **año anterior** (1934) **sin necesidad de realizar un JOIN complejo**.

Para que esta comparación funcione correctamente:

- La **ventana** debe estar siempre **ordenada cronológicamente**.
- Se utiliza `ORDER BY` para definir ese orden temporal.

---

### 🎯 Objetivo del ejercicio

Comparar la **cantidad de goles anotados** en cada mundial con respecto al **mundial anterior**, con el fin de calcular el **crecimiento o la caída** de goles a lo largo de la historia.


```SQL
SELECT
    year,
    country,
    goalsscored,
    LAG(goalsscored) OVER(ORDER BY year ASC) AS goles_anteriores,
    goalsscored - LAG(goalsscored) OVER (ORDER BY year ASC) AS diferencia
FROM worldcup.cups
```

In [25]:
df_c = df_cups.copy().sort_values(by='year')

df_c['goles_anteriores'] = df_c['goalsscored'].shift(1).fillna(0).astype(int)

df_c['diferencia'] = df_c['goalsscored'] - df_c['goles_anteriores']

res = df_c[['year','country', 'goalsscored', 'goles_anteriores', 'diferencia']]

In [29]:
res = pl_cups.sort('year').with_columns([
    pl.col('goalsscored')
    .shift(1)
    .fill_null(0)
    .alias('goles_anteriores')
]).with_columns([
(pl.col('goalsscored') - pl.col('goles_anteriores')).alias('diferencia')
]).select([
    'year', 'country', 'goalsscored', 'goles_anteriores', 'diferencia'
])

## 🛠️ Ejercicio 5.3: Mirando al Futuro (LEAD)

### 📐 La lógica de la anticipación

La función **`LEAD()`** permite **mirar hacia adelante** en el tiempo.  
Por ejemplo, estando en la fila del mundial de **1930**, puedes ver en esa misma fila cuántos goles (o equipos) se registrarán en el **siguiente mundial** (**1934**).

Este enfoque evita la necesidad de realizar **joins temporales** complejos.

---

### 🎯 Objetivo del ejercicio

Crear un **reporte** que muestre:

- El **año actual** del mundial.
- La cantidad de **equipos clasificados** (`QualifiedTeams`) en el **siguiente mundial**.

El objetivo es analizar si la **FIFA planeaba expandir el torneo** a lo largo del tiempo.

```SQL
SELECT
    year,
    country,
    qualifiedteams AS equipos_actuales,
    LEAD(qualifiedteams) OVER(ORDER BY year ASC) AS equipos_proximo_mundial
FROM worldcup.cups;
```

In [32]:
df_c = df_cups.copy().sort_values('year')

df_c['equipos_proximo'] = df_c['qualifiedteams'].shift(-1).fillna(0).astype(int)

res =  df_c[['year','country','qualifiedteams','equipos_proximo']]

In [33]:
res = pl_cups.sort('year').with_columns([
    pl.col('qualifiedteams')
    .shift(-1)
    .fill_null(0)
    .alias('equipos_proximo')
]).select([
    'year', 'country', 'qualifiedteams', 'equipos_proximo'
])

res

year,country,qualifiedteams,equipos_proximo
i64,str,i64,i64
1930,"""Uruguay""",13,16
1934,"""Italy""",16,15
1938,"""France""",15,13
1950,"""Brazil""",13,16
1954,"""Switzerland""",16,16
…,…,…,…
1998,"""France""",32,32
2002,"""Korea/Japan""",32,32
2006,"""Germany""",32,32
